# Modelo de Ising con campo transverso y como simularlo

Imaginemos que tenemos un arreglo de Spines bidimensional en un plano perpendicular al eje Z. Y este arreglo tiene la propiedad de que hay un campo magnético que esta apuntando en la dirección X. ¿Como se visualiza esto en un Hamiltoniano? La respuesta esta codificada en el siguiente Hamiltoniano:

$$ H=-J\sum_i Z_iZ_{i+1}-h\sum_i X_i$$

Aquí tenemos 2 términos: 

- El primero busca que los espines se ordenen en z, es decir, que se emparejen spines similares. 
- El segundo quiere que cada espín apunte a x, pero dado que las mediciones se hacen en la base computacional, es decir, sobre z, esto es equivalente a hacerlos caer en superposición

## Recordatorio: Intuición detras del Ising clásico

En el Ising clásico, a cada spin le asignamos una variable $s_i$ tal que $s_i \in \{+1, -1\}$. Esto puede representar que el Spin esta apuntando hacia arriba (si positivo) o abajo (si negativo). Ahora, hay 2 posibles términos que nos pueden dar la energía del sistema:

- **Acoplamiento**: $E_{acopl}=-J\sum_i s_i s_{i+1}$. Esta energía compara a los vecinos y decide si vale la pena que se emparejen o no. Si $J>0$, entonces tienden a alinearse para menor energía (regimen ferromagnético). Caso contrario, el sistema prefiere alternancia (Regimen antiferromagnetico).
- **Campo Magnético externo**: $E_{m}=h\sum s_i$ Si hay una fuerza externa apuntando a alguno de los ejes, esta altera a cada una de las variables en proporción a la misma variable. Entonces


El Hamiltoniano resulta ser la suma de todas las energía del sistema. 

## Hamiltoniano Cuántico

Por el momento tomemos el Ising clásico mas simple: Modelo solo con Acoplamiento. El modelo de Ising asigna variables clásicas al sistema de spines. Pero esto asume que los spines son entidades matemáticas clásicas bien definidas, y no es el caso siempre. Entonces podemos ver los spines como estados cuánticos, donde $|0\rangle$ es apuntar abajo y $|1\rangle$ hacia arriba. Con esto en mente, lo que se haces es en el Hamilotniano cambias la variable $s_i$ por el operador $Z_i$, es decir, la matri de paulli Z aplicada sobre el i-esimo qubit.

$$H_{ZZ}=-J\sum_i Z_i Z_{i+1}$$

## Campo Transverso

Ahora, imaginemos que hay un campo magnético pero que en esta ocasión es paralelo al eje X. Entonces el Hamiltoniano que lo modela es:

$$H_X = -h\sum_i X_i$$

Con esto es que recuperamos la fórmula completa para el campo transverso:

$$ H= H_{ZZ}+H_X=-J\sum_i Z_iZ_{i+1}-h\sum_i X_i$$

Quiero que notes como ambos modelos quieren 2 cosas distintas. El primero quiere acoplar los atomos de una forma concreta, el segundo quiere introducir un superposición al forzar al spin a caer en X y medir en Z. Ademas, notemos que los operadores Z no conmutan con X, por lo cual para correrlos necesitamos Trotterización

## Evolución temporal y problemas

El operador de evolución temporal tiene un gran problema. Recordemos que

$$|\psi(t)\rangle = e^{-iHt}|\psi(0)\rangle$$

y sustituyendo el Hamiltoniano del Ising con campo transverso:

$$|\psi(t)\rangle = e^{-i (H_{ZZ}+H_X) t}$$

Este problema requiere Trotterización, pero veremos como evoluciona con el tiempo, poco a poco. Antes, analicemos lo que el modelo nos esta diciendo

## Condiciones de Frontera

Existen 2 posibilidades para las condiciones de Frontera del problema de Ising 1D:

- Abierta: Donde la cadena de spines tiene inicio y fin, y el ultimo spin y el primero estan separados

- Cerrada: Donde la cadena de spines es un ciclo en donde el ultimo y el primer spin estan en contacto el uno con el otro

## Analisis de limites

¿Que pasa si $J\gg h$?

Notemos que si $J>0$, los estados de mínima energía son aprox $|00...0\rangle$ y $|11...1\rangle$. En este caso, la correlación, es decir $\langle Z_i Z_{i+1} \rangle \approx 1$

¿Que pasa si $h\gg J$?

El hamiltoniano se vuelve aprox $H\approx -h\sum_i X_i$, Lo cual hace que casa qubit en $h>0$ quiera estar en $|+\rangle$

## Simulación del Hamiltoniano clásica

Podemos resolver el sistema de manera completamente clásica inicialmente. Para ello, entendamos como le hacemos

Primero, construimos la matriz del Hamiltoniano utilizando productos de Kronecker para el producto tensorial de los sistemas y luego sumando los valores. 

Una vez encontrdo H de forma directa, despues por el teorema de diagonalización exacta, al ser H un operador hermitiano, lo podemos descomponer en $H=VDV^{\dagger}$ en donde D contiene los eigenvalores $E_k$ de H como elementos de su diagonal y las columnas de $V$ son los aigenvectores $|E_k\rangle$, convirtiendose en una matriz de cambio de base. Esto tiene la siguiente propiedad:

$$e^{-iHt}=Ve^{-iDT}V^{\dagger}$$

Ahora, al ser D diagonal, ocurre que $e^{-iDt}=diag(e^{-iE_0t},e^{-iE_1t},...)$

Y al final $|\psi(t)\rangle = Ve^{-iDt}V^{\dagger}|\psi(0)\rangle$

Veamoslo en código

In [6]:
import numpy as np

from scipy.linalg import eigh
from qiskit.quantum_info import SparsePauliOp, Statevector


def evolve_tfim_exact(
    t: float,
    h: float,
    J: float,
    n_qubits: int,
    r: int,
    periodic: bool = False,
) -> Statevector:

    if n_qubits < 1:
        raise ValueError("n_qubits debe ser al menos 1.")

    if r < 1:
        raise ValueError("r debe ser al menos 1.")

    if t < 0:
        raise ValueError("t debe ser mayor o igual que cero.")

    if periodic and n_qubits < 2:
        raise ValueError(
            "Las condiciones periódicas requieren al menos 2 qubits."
        )

    # Cada término se guarda como:
    #
    #     ("cadena de Pauli", coeficiente)
    #
    # Por ejemplo:
    #     ("ZZII", -J)
    #     ("IIXI", -h)
    # En Qiskit, el qubit 0 aparece en el extremo derecho de
    # la cadena de Pauli.

    pauli_terms = []

    # Términos del campo transversal:
    #     -h Σ_i X_i

    for qubit in range(n_qubits):
        label = ["I"] * n_qubits

        # el qubit 0 corresponde al carácter más a la derecha.
        label[n_qubits - 1 - qubit] = "X"

        pauli_string = "".join(label)

        pauli_terms.append((pauli_string, -h))

    # Términos de interacción ZZ

    if periodic and n_qubits > 2:

        # Frontera periódica:
        # (0,1), (1,2), ..., (N-2,N-1), (N-1,0)

        pairs = [
            (qubit, (qubit + 1) % n_qubits)
            for qubit in range(n_qubits)
        ]

    else:

        # Frontera abierta:
        # (0,1), (1,2), ..., (N-2,N-1)
        # Para N=2, la frontera periódica contiene físicamente
        # el mismo enlace que la frontera abierta, así que no
        # debe agregarse dos veces.

        pairs = [
            (qubit, qubit + 1)
            for qubit in range(n_qubits - 1)
        ]

    for qubit_1, qubit_2 in pairs:
        label = ["I"] * n_qubits

        label[n_qubits - 1 - qubit_1] = "Z"
        label[n_qubits - 1 - qubit_2] = "Z"

        pauli_string = "".join(label)

        pauli_terms.append((pauli_string, -J))

    # Construcción del Hamiltoniano

    hamiltonian_operator = SparsePauliOp.from_list(pauli_terms) 

    # Convertimos el operador de Pauli en una matriz densa
    # de dimensión 2^N × 2^N.
    hamiltonian_matrix = hamiltonian_operator.to_matrix()

    # ---------------------------------------------------------
    # Diagonalización exacta
    # ---------------------------------------------------------
    #
    # eigh se utiliza porque H es hermitiano.
    #
    # eigenvalues:
    #     E_0, E_1, ..., E_(2^N - 1)
    #
    # eigenvectors:
    #     matriz cuyas columnas son |E_k>
    # ---------------------------------------------------------

    eigenvalues, eigenvectors = eigh(hamiltonian_matrix)

    # ---------------------------------------------------------
    # Estado inicial |00...0>
    # ---------------------------------------------------------

    dimension = 2**n_qubits

    initial_state = np.zeros(dimension, dtype=complex)
    initial_state[0] = 1.0

    # ---------------------------------------------------------
    # Expresamos el estado inicial en la base de energía:
    #
    #     coefficients = V† |ψ(0)>
    # ---------------------------------------------------------

    coefficients = eigenvectors.conj().T @ initial_state

    # ---------------------------------------------------------
    # Cada componente de energía adquiere una fase:
    #
    #     c_k(t) = exp(-i E_k t) c_k(0)
    # ---------------------------------------------------------

    evolved_coefficients = (
        np.exp(-1j * eigenvalues * t) * coefficients
    )

    # ---------------------------------------------------------
    # Regresamos de la base de energía a la base computacional:
    #
    #     |ψ(t)> = V c(t)
    # ---------------------------------------------------------

    evolved_state = eigenvectors @ evolved_coefficients

    # Se devuelve como Statevector para que sea compatible
    # con la función cuántica trotterizada.
    return Statevector(evolved_state)

In [7]:
exact_state_open = evolve_tfim_exact(
    t=1.0,
    h=0.5,
    J=1.0,
    n_qubits=4,
    r=10,
    periodic=False,
)

print(exact_state_open)

Statevector([-0.79694162-1.11438641e-01j, -0.26329734-1.98153060e-01j,
             -0.13269034+4.86937038e-02j, -0.00517471-1.46154522e-01j,
             -0.13269034+4.86937038e-02j, -0.05494956+2.98372438e-16j,
             -0.10565793-2.25021404e-02j,  0.04349908-4.59123428e-02j,
             -0.26329734-1.98153060e-01j, -0.05401606-1.23652381e-01j,
             -0.05494956+3.08780779e-16j,  0.02279121-5.60726974e-02j,
             -0.00517471-1.46154522e-01j,  0.02279121-5.60726974e-02j,
              0.04349908-4.59123428e-02j,  0.01641573+3.22554259e-02j],
            dims=(2, 2, 2, 2))


## Trotterización del Hamiltoniano TFIM

En el notebook anterior vimos que, si un Hamiltoniano puede escribirse como

$$
H=A+B,
$$

podemos aproximar su evolución temporal mediante la fórmula de Suzuki-Trotter.

Para el Modelo de Ising de Campo Transverso,

$$
H=
-J\sum_{i=0}^{N-2}Z_iZ_{i+1}
-h\sum_{i=0}^{N-1}X_i,
$$

identificamos

$$
H_{ZZ}
=
-J\sum_{i=0}^{N-2}Z_iZ_{i+1},
$$

$$
H_X
=
-h\sum_{i=0}^{N-1}X_i.
$$

Dividiendo el tiempo total en $r$ pasos,

$$
\Delta t=\frac{t}{r},
$$

la aproximación de primer orden queda

$$
U(t)
\approx
\left(
e^{-iH_{ZZ}\Delta t}
e^{-iH_X\Delta t}
\right)^r.
$$

Como todos los términos $Z_iZ_{i+1}$ conmutan entre sí, y todos los $X_i$ también, cada exponencial puede implementarse de forma independiente:

$$
e^{-iH_{ZZ}\Delta t}
=
\prod_{i=0}^{N-2}
e^{\,iJ\Delta t\,Z_iZ_{i+1}},
$$

$$
e^{-iH_X\Delta t}
=
\prod_{i=0}^{N-1}
e^{\,ih\Delta t\,X_i}.
$$

Finalmente, cada exponencial se implementa mediante rotaciones:

$$
e^{\,ih\Delta tX}
=
R_X(-2h\Delta t),
$$

$$
e^{\,iJ\Delta tZ_iZ_{i+1}}
=
R_{ZZ}(-2J\Delta t).
$$

Por tanto, **cada paso de Trotter** consiste simplemente en:

1. Aplicar una compuerta $R_{ZZ}(-2J\Delta t)$ entre todos los pares de vecinos.
2. Aplicar una compuerta $R_X(-2h\Delta t)$ sobre todos los qubits.
3. Repetir el procedimiento $r$ veces.

Ejercicio Incial Mariana y Amado:

Trottericen para el Hamiltoniano de Ising con campo magnetico transversal. Haganlo para que se pueda generalizar a n qubits



Funcion (J, h, Num qubits, r,t) nos debe devolver un estado final

In [ ]:
def H_trott1_c(J: float, h: float, n_q: int, r: int, t: float) -> Statevector:
    dt = t/r
    qc = QuantumCircuit(n_q)

    # trotterization
    for _ in range(r):
        for i in range(n_q):
            qc.rzz(-2*J*dt, i-1, i)
            qc.rx(-2*h*dt, i-1)

    return Statevector.from_instruction(qc)

def H_trott1_o(J: float, h: float, n_q: int, r: int, t: float) -> Statevector:
    dt = t/r
    qc = QuantumCircuit(n_q)

    # trotterization
    for _ in range(r):
        for i in range(1, n_q):
            qc.rzz(-2*J*dt, i-1, i)
            qc.rx(-2*h*dt, i-1)
        qc.rx(-2*h*dt, n_q-1)

    return Statevector.from_instruction(qc)